# AE101 매일 배치 노트북 (스케줄 실행용)

이 노트북은 **사람이 아니라 태스크가 매일 실행**합니다. 설치는 `orchestration/daily_schedule.sql` 을 보세요.
워크스루 노트북(`ae101_enterprise_walkthrough.ipynb`)의 STEP 2 ~ STEP 7 셀을 글자 그대로 가져왔습니다. 로직을 바꿀 때는 워크스루를 고치고 `python scripts/build_daily_notebook.py` 로 다시 만드세요.

| 넣은 단계 | 넣지 않은 단계 (한 번만 또는 수동) |
|---|---|
| Git FETCH → STEP 2 실행 시작 → STEP 3 RAW 새 배치 → STEP 4~5 CORE → STEP 6 감사 기록 → STEP 7 MART 반영·품질 검사·상태 확정 | STEP 1 DDL, STEP 8 리포트 뷰, STEP 9 보안, STEP 10 확인 쿼리, STEP 11 SCD2 체험, STEP 12 정리 |

**사전 조건** : 워크스루 STEP 1(DDL)과 STEP 8(리포트 뷰)을 한 번 실행했어야 합니다.

**실패 처리** : 정상 주문이 0건이면 `daily_guard` 가, 품질 검사가 실패하면 `daily_gate` 가 오류를 내 실행을 실패로 끝냅니다. 중간 셀에서 오류가 나 `RUNNING` 으로 남은 실행은 파이널라이저 태스크가 `FAILED` 로 바꿉니다.

**주의** : 스케줄이 켜져 있는 동안 워크스루 노트북을 직접 실행하면 두 실행이 같은 `CORE.RUN_CTX` 를 덮어씁니다. 실습 전에 `ALTER TASK AE101_SALES_ENT.OPS.T_DAILY_BATCH SUSPEND;` 로 멈추세요.

## 0. 준비 : DB 지정, Git 저장소 최신화

In [ ]:
-- 태스크는 노트북 소유 역할로 실행됩니다. USE ROLE 로 역할을 바꾸지 않습니다.
USE DATABASE AE101_SALES_ENT;
ALTER GIT REPOSITORY AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE FETCH;

## STEP 2. 실행 시작 : load_id / run_ts 발급, RUNNING 기록

In [ ]:
-- UUID_STRING() 같은 비결정 함수는 SET 에 바로 쓸 수 없는 환경(노트북 등)이 있어, 먼저 테이블에 발급한 뒤 변수로 읽습니다.
CREATE OR REPLACE TABLE CORE.RUN_CTX AS
  SELECT UUID_STRING() AS load_id, CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS run_ts;
SET load_id = (SELECT load_id FROM CORE.RUN_CTX);
SET run_ts  = (SELECT run_ts  FROM CORE.RUN_CTX);

INSERT INTO AUDIT.PIPELINE_RUNS (load_id, status, started_at) SELECT $load_id, 'RUNNING', $run_ts;

## STEP 3. RAW 적재 : Git 저장소 CSV 를 새 배치로 추가

In [ ]:
INSERT INTO RAW.CUSTOMERS (load_id, loaded_at, src_file, src_row, customer_id, customer_name, email, region)
SELECT $load_id, $run_ts, 'customers.csv', ROW_NUMBER() OVER (ORDER BY METADATA$FILE_ROW_NUMBER) + 1, $1, $2, $3, $4
FROM @AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE/branches/main/data/raw/customers.csv (FILE_FORMAT => 'AE101_SALES_ENT.RAW.CSV_FORMAT');

INSERT INTO RAW.PRODUCTS (load_id, loaded_at, src_file, src_row, product_id, product_name, category)
SELECT $load_id, $run_ts, 'products.csv', ROW_NUMBER() OVER (ORDER BY METADATA$FILE_ROW_NUMBER) + 1, $1, $2, $3
FROM @AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE/branches/main/data/raw/products.csv (FILE_FORMAT => 'AE101_SALES_ENT.RAW.CSV_FORMAT');

INSERT INTO RAW.ORDERS (load_id, loaded_at, src_file, src_row, order_id, customer_id, order_date, order_status, order_amount)
SELECT $load_id, $run_ts, 'orders.csv', ROW_NUMBER() OVER (ORDER BY METADATA$FILE_ROW_NUMBER) + 1, $1, $2, $3, $4, $5
FROM @AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE/branches/main/data/raw/orders.csv (FILE_FORMAT => 'AE101_SALES_ENT.RAW.CSV_FORMAT');

INSERT INTO RAW.ORDER_ITEMS (load_id, loaded_at, src_file, src_row, order_item_id, order_id, product_id, quantity, unit_price)
SELECT $load_id, $run_ts, 'order_items.csv', ROW_NUMBER() OVER (ORDER BY METADATA$FILE_ROW_NUMBER) + 1, $1, $2, $3, $4, $5
FROM @AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE/branches/main/data/raw/order_items.csv (FILE_FORMAT => 'AE101_SALES_ENT.RAW.CSV_FORMAT');

INSERT INTO RAW.PAYMENTS (load_id, loaded_at, src_file, src_row, payment_id, order_id, payment_status, payment_amount, paid_at)
SELECT $load_id, $run_ts, 'payments.csv', ROW_NUMBER() OVER (ORDER BY METADATA$FILE_ROW_NUMBER) + 1, $1, $2, $3, $4, $5
FROM @AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE/branches/main/data/raw/payments.csv (FILE_FORMAT => 'AE101_SALES_ENT.RAW.CSV_FORMAT');

-- 확인: 배치별 행 수 (최신 배치 합계 129,618 = 고객 2,028 + 상품 151 + 주문 30,081 + 상세 66,262 + 결제 31,096)
SELECT 'customers' AS file, load_id, COUNT(*) AS row_count FROM RAW.CUSTOMERS GROUP BY load_id
UNION ALL SELECT 'products',    load_id, COUNT(*) FROM RAW.PRODUCTS    GROUP BY load_id
UNION ALL SELECT 'orders',      load_id, COUNT(*) FROM RAW.ORDERS      GROUP BY load_id
UNION ALL SELECT 'order_items', load_id, COUNT(*) FROM RAW.ORDER_ITEMS GROUP BY load_id
UNION ALL SELECT 'payments',    load_id, COUNT(*) FROM RAW.PAYMENTS    GROUP BY load_id;

## STEP 4. CORE 정제

In [ ]:
-- >>> LOGIC START
CREATE OR REPLACE TABLE CORE.ISSUES_WORK (
  source_file VARCHAR, source_row_number NUMBER, record_key VARCHAR,
  error_code VARCHAR, error_detail VARCHAR, order_id VARCHAR, action VARCHAR);

-- ---------- customers (email·region 선택값, region 없으면 'Unknown') ----------
CREATE OR REPLACE TABLE CORE.CUSTOMERS_CHK AS
WITH d AS (
  SELECT r.src_row,
         NULLIF(TRIM(r.customer_id), '')   AS customer_id,
         NULLIF(TRIM(r.customer_name), '') AS customer_name,
         NULLIF(TRIM(r.email), '')         AS email,
         NULLIF(TRIM(r.region), '')        AS region,
         ROW_NUMBER() OVER (PARTITION BY r.customer_id, r.customer_name, r.email, r.region ORDER BY r.src_row) AS rn
  FROM RAW.V_CUSTOMERS_LATEST r),
u AS (
  SELECT d.*, COUNT(*) OVER (PARTITION BY d.customer_id) AS key_cnt,
         CASE WHEN d.customer_id   IS NULL THEN 'customer_id: required value missing'
              WHEN d.customer_name IS NULL THEN 'customer_name: required value missing' END AS err
  FROM d WHERE d.rn = 1)
SELECT src_row, customer_id, customer_name, email, region, 'DUPLICATE' AS dq_status, CAST(NULL AS VARCHAR) AS err FROM d WHERE rn > 1
UNION ALL
SELECT src_row, customer_id, customer_name, email, region,
       CASE WHEN customer_id IS NOT NULL AND key_cnt > 1 THEN 'CONFLICT' WHEN err IS NOT NULL THEN 'INVALID' ELSE 'OK' END, err
FROM u;

INSERT INTO CORE.ISSUES_WORK
SELECT 'customers.csv', src_row, customer_id,
       CASE dq_status WHEN 'DUPLICATE' THEN 'EXACT_DUPLICATE' WHEN 'CONFLICT' THEN 'KEY_CONFLICT' ELSE 'INVALID_VALUE' END,
       CASE dq_status WHEN 'DUPLICATE' THEN 'Identical source row removed' WHEN 'CONFLICT' THEN 'Same business key has conflicting source rows' ELSE err END,
       NULL, CASE dq_status WHEN 'DUPLICATE' THEN 'REMOVED' ELSE 'QUARANTINED' END
FROM CORE.CUSTOMERS_CHK WHERE dq_status <> 'OK';

CREATE OR REPLACE TABLE CORE.CUSTOMERS AS
SELECT customer_id, customer_name, email, COALESCE(region, 'Unknown') AS region,
       MD5(COALESCE(customer_name, '') || '|' || COALESCE(email, '') || '|' || COALESCE(region, 'Unknown')) AS hash_diff,
       src_row
FROM CORE.CUSTOMERS_CHK WHERE dq_status = 'OK';

-- ---------- products ----------
CREATE OR REPLACE TABLE CORE.PRODUCTS_CHK AS
WITH d AS (
  SELECT r.src_row,
         NULLIF(TRIM(r.product_id), '')   AS product_id,
         NULLIF(TRIM(r.product_name), '') AS product_name,
         NULLIF(TRIM(r.category), '')     AS category,
         ROW_NUMBER() OVER (PARTITION BY r.product_id, r.product_name, r.category ORDER BY r.src_row) AS rn
  FROM RAW.V_PRODUCTS_LATEST r),
u AS (
  SELECT d.*, COUNT(*) OVER (PARTITION BY d.product_id) AS key_cnt,
         CASE WHEN d.product_id   IS NULL THEN 'product_id: required value missing'
              WHEN d.product_name IS NULL THEN 'product_name: required value missing'
              WHEN d.category     IS NULL THEN 'category: required value missing' END AS err
  FROM d WHERE d.rn = 1)
SELECT src_row, product_id, product_name, category, 'DUPLICATE' AS dq_status, CAST(NULL AS VARCHAR) AS err FROM d WHERE rn > 1
UNION ALL
SELECT src_row, product_id, product_name, category,
       CASE WHEN product_id IS NOT NULL AND key_cnt > 1 THEN 'CONFLICT' WHEN err IS NOT NULL THEN 'INVALID' ELSE 'OK' END, err
FROM u;

INSERT INTO CORE.ISSUES_WORK
SELECT 'products.csv', src_row, product_id,
       CASE dq_status WHEN 'DUPLICATE' THEN 'EXACT_DUPLICATE' WHEN 'CONFLICT' THEN 'KEY_CONFLICT' ELSE 'INVALID_VALUE' END,
       CASE dq_status WHEN 'DUPLICATE' THEN 'Identical source row removed' WHEN 'CONFLICT' THEN 'Same business key has conflicting source rows' ELSE err END,
       NULL, CASE dq_status WHEN 'DUPLICATE' THEN 'REMOVED' ELSE 'QUARANTINED' END
FROM CORE.PRODUCTS_CHK WHERE dq_status <> 'OK';

CREATE OR REPLACE TABLE CORE.PRODUCTS AS
SELECT product_id, product_name, category,
       MD5(COALESCE(product_name, '') || '|' || COALESCE(category, '')) AS hash_diff,
       src_row
FROM CORE.PRODUCTS_CHK WHERE dq_status = 'OK';

-- ---------- orders ----------
CREATE OR REPLACE TABLE CORE.ORDERS_CHK AS
WITH d AS (
  SELECT r.src_row,
         NULLIF(TRIM(r.order_id), '')     AS order_id,
         NULLIF(TRIM(r.customer_id), '')  AS customer_id,
         NULLIF(TRIM(r.order_date), '')   AS order_date,
         NULLIF(TRIM(r.order_status), '') AS order_status,
         NULLIF(TRIM(r.order_amount), '') AS order_amount,
         ROW_NUMBER() OVER (PARTITION BY r.order_id, r.customer_id, r.order_date, r.order_status, r.order_amount ORDER BY r.src_row) AS rn
  FROM RAW.V_ORDERS_LATEST r),
u AS (
  SELECT d.*, COUNT(*) OVER (PARTITION BY d.order_id) AS key_cnt,
         CASE WHEN d.order_id     IS NULL THEN 'order_id: required value missing'
              WHEN d.customer_id  IS NULL THEN 'customer_id: required value missing'
              WHEN d.order_date   IS NULL THEN 'order_date: required value missing'
              WHEN d.order_status IS NULL THEN 'order_status: required value missing'
              WHEN d.order_amount IS NULL THEN 'order_amount: required value missing'
              WHEN NOT REGEXP_LIKE(d.order_amount, '[0-9]+') THEN 'order_amount: expected non-negative integer KRW/quantity'
              WHEN TRY_TO_NUMBER(d.order_amount) > 1000000000000 THEN 'order_amount: exceeds project input limit'
              WHEN TRY_TO_DATE(d.order_date, 'YYYY-MM-DD') IS NULL THEN 'order_date: invalid date'
              WHEN d.order_status NOT IN ('completed', 'pending', 'cancelled') THEN 'unknown order_status' END AS err
  FROM d WHERE d.rn = 1)
SELECT src_row, order_id, customer_id, order_date, order_status, order_amount, 'DUPLICATE' AS dq_status, CAST(NULL AS VARCHAR) AS err FROM d WHERE rn > 1
UNION ALL
SELECT src_row, order_id, customer_id, order_date, order_status, order_amount,
       CASE WHEN order_id IS NOT NULL AND key_cnt > 1 THEN 'CONFLICT' WHEN err IS NOT NULL THEN 'INVALID' ELSE 'OK' END, err
FROM u;

INSERT INTO CORE.ISSUES_WORK
SELECT 'orders.csv', src_row, order_id,
       CASE dq_status WHEN 'DUPLICATE' THEN 'EXACT_DUPLICATE' WHEN 'CONFLICT' THEN 'KEY_CONFLICT' ELSE 'INVALID_VALUE' END,
       CASE dq_status WHEN 'DUPLICATE' THEN 'Identical source row removed' WHEN 'CONFLICT' THEN 'Same business key has conflicting source rows' ELSE err END,
       order_id, CASE dq_status WHEN 'DUPLICATE' THEN 'REMOVED' ELSE 'QUARANTINED' END
FROM CORE.ORDERS_CHK WHERE dq_status <> 'OK';

CREATE OR REPLACE TABLE CORE.ORDERS AS
SELECT c.order_id, c.customer_id, TO_DATE(c.order_date, 'YYYY-MM-DD') AS order_date, c.order_status,
       TO_NUMBER(c.order_amount) AS order_amount,
       c.customer_id IN (SELECT customer_id FROM CORE.CUSTOMERS) AS has_valid_customer,
       c.src_row
FROM CORE.ORDERS_CHK c WHERE c.dq_status = 'OK';

INSERT INTO CORE.ISSUES_WORK
SELECT 'orders.csv', src_row, order_id, 'INVALID_REFERENCE', 'Unknown customer: loaded with customer_key = -1', order_id, 'FLAGGED'
FROM CORE.ORDERS WHERE NOT has_valid_customer;

-- ---------- order_items ----------
CREATE OR REPLACE TABLE CORE.ORDER_ITEMS_CHK AS
WITH d AS (
  SELECT r.src_row,
         NULLIF(TRIM(r.order_item_id), '') AS order_item_id,
         NULLIF(TRIM(r.order_id), '')      AS order_id,
         NULLIF(TRIM(r.product_id), '')    AS product_id,
         NULLIF(TRIM(r.quantity), '')      AS quantity,
         NULLIF(TRIM(r.unit_price), '')    AS unit_price,
         ROW_NUMBER() OVER (PARTITION BY r.order_item_id, r.order_id, r.product_id, r.quantity, r.unit_price ORDER BY r.src_row) AS rn
  FROM RAW.V_ORDER_ITEMS_LATEST r),
u AS (
  SELECT d.*, COUNT(*) OVER (PARTITION BY d.order_item_id) AS key_cnt,
         CASE WHEN d.order_item_id IS NULL THEN 'order_item_id: required value missing'
              WHEN d.order_id      IS NULL THEN 'order_id: required value missing'
              WHEN d.product_id    IS NULL THEN 'product_id: required value missing'
              WHEN d.quantity      IS NULL THEN 'quantity: required value missing'
              WHEN d.unit_price    IS NULL THEN 'unit_price: required value missing'
              WHEN NOT REGEXP_LIKE(d.quantity, '[0-9]+') THEN 'quantity: expected non-negative integer KRW/quantity'
              WHEN TRY_TO_NUMBER(d.quantity) > 1000000000000 THEN 'quantity: exceeds project input limit'
              WHEN NOT REGEXP_LIKE(d.unit_price, '[0-9]+') THEN 'unit_price: expected non-negative integer KRW/quantity'
              WHEN TRY_TO_NUMBER(d.unit_price) > 1000000000000 THEN 'unit_price: exceeds project input limit'
              WHEN TRY_TO_NUMBER(d.quantity) = 0 THEN 'quantity must be positive' END AS err
  FROM d WHERE d.rn = 1)
SELECT src_row, order_item_id, order_id, product_id, quantity, unit_price, 'DUPLICATE' AS dq_status, CAST(NULL AS VARCHAR) AS err FROM d WHERE rn > 1
UNION ALL
SELECT src_row, order_item_id, order_id, product_id, quantity, unit_price,
       CASE WHEN order_item_id IS NOT NULL AND key_cnt > 1 THEN 'CONFLICT' WHEN err IS NOT NULL THEN 'INVALID' ELSE 'OK' END, err
FROM u;

INSERT INTO CORE.ISSUES_WORK
SELECT 'order_items.csv', src_row, order_item_id,
       CASE dq_status WHEN 'DUPLICATE' THEN 'EXACT_DUPLICATE' WHEN 'CONFLICT' THEN 'KEY_CONFLICT' ELSE 'INVALID_VALUE' END,
       CASE dq_status WHEN 'DUPLICATE' THEN 'Identical source row removed' WHEN 'CONFLICT' THEN 'Same business key has conflicting source rows' ELSE err END,
       order_id, CASE dq_status WHEN 'DUPLICATE' THEN 'REMOVED' ELSE 'QUARANTINED' END
FROM CORE.ORDER_ITEMS_CHK WHERE dq_status <> 'OK';

-- 주문이 없는 상세는 날짜·고객을 알 수 없으므로 격리
INSERT INTO CORE.ISSUES_WORK
SELECT 'order_items.csv', src_row, order_item_id, 'INVALID_REFERENCE', 'Parent order not found: quarantined', order_id, 'QUARANTINED'
FROM CORE.ORDER_ITEMS_CHK
WHERE dq_status = 'OK' AND order_id NOT IN (SELECT order_id FROM CORE.ORDERS);

CREATE OR REPLACE TABLE CORE.ORDER_ITEMS AS
SELECT c.order_item_id, c.order_id, c.product_id,
       TO_NUMBER(c.quantity) AS quantity, TO_NUMBER(c.unit_price) AS unit_price,
       c.product_id IN (SELECT product_id FROM CORE.PRODUCTS) AS has_valid_product,
       c.src_row
FROM CORE.ORDER_ITEMS_CHK c
WHERE c.dq_status = 'OK' AND c.order_id IN (SELECT order_id FROM CORE.ORDERS);

INSERT INTO CORE.ISSUES_WORK
SELECT 'order_items.csv', src_row, order_item_id, 'INVALID_REFERENCE', 'Unknown product: loaded with product_key = -1', order_id, 'FLAGGED'
FROM CORE.ORDER_ITEMS WHERE NOT has_valid_product;

-- ---------- payments ----------
CREATE OR REPLACE TABLE CORE.PAYMENTS_CHK AS
WITH d AS (
  SELECT r.src_row,
         NULLIF(TRIM(r.payment_id), '')     AS payment_id,
         NULLIF(TRIM(r.order_id), '')       AS order_id,
         NULLIF(TRIM(r.payment_status), '') AS payment_status,
         NULLIF(TRIM(r.payment_amount), '') AS payment_amount,
         NULLIF(TRIM(r.paid_at), '')        AS paid_at,
         ROW_NUMBER() OVER (PARTITION BY r.payment_id, r.order_id, r.payment_status, r.payment_amount, r.paid_at ORDER BY r.src_row) AS rn
  FROM RAW.V_PAYMENTS_LATEST r),
u AS (
  SELECT d.*, COUNT(*) OVER (PARTITION BY d.payment_id) AS key_cnt,
         CASE WHEN d.payment_id     IS NULL THEN 'payment_id: required value missing'
              WHEN d.order_id       IS NULL THEN 'order_id: required value missing'
              WHEN d.payment_status IS NULL THEN 'payment_status: required value missing'
              WHEN d.payment_amount IS NULL THEN 'payment_amount: required value missing'
              WHEN d.paid_at        IS NULL THEN 'paid_at: required value missing'
              WHEN NOT REGEXP_LIKE(d.payment_amount, '[0-9]+') THEN 'payment_amount: expected non-negative integer KRW/quantity'
              WHEN TRY_TO_NUMBER(d.payment_amount) > 1000000000000 THEN 'payment_amount: exceeds project input limit'
              WHEN d.payment_status NOT IN ('succeeded', 'failed', 'pending') THEN 'unknown payment_status'
              WHEN TRY_TO_TIMESTAMP_NTZ(d.paid_at, 'YYYY-MM-DD HH24:MI:SS') IS NULL THEN 'paid_at: invalid timestamp' END AS err
  FROM d WHERE d.rn = 1)
SELECT src_row, payment_id, order_id, payment_status, payment_amount, paid_at, 'DUPLICATE' AS dq_status, CAST(NULL AS VARCHAR) AS err FROM d WHERE rn > 1
UNION ALL
SELECT src_row, payment_id, order_id, payment_status, payment_amount, paid_at,
       CASE WHEN payment_id IS NOT NULL AND key_cnt > 1 THEN 'CONFLICT' WHEN err IS NOT NULL THEN 'INVALID' ELSE 'OK' END, err
FROM u;

INSERT INTO CORE.ISSUES_WORK
SELECT 'payments.csv', src_row, payment_id,
       CASE dq_status WHEN 'DUPLICATE' THEN 'EXACT_DUPLICATE' WHEN 'CONFLICT' THEN 'KEY_CONFLICT' ELSE 'INVALID_VALUE' END,
       CASE dq_status WHEN 'DUPLICATE' THEN 'Identical source row removed' WHEN 'CONFLICT' THEN 'Same business key has conflicting source rows' ELSE err END,
       order_id, CASE dq_status WHEN 'DUPLICATE' THEN 'REMOVED' ELSE 'QUARANTINED' END
FROM CORE.PAYMENTS_CHK WHERE dq_status <> 'OK';

CREATE OR REPLACE TABLE CORE.PAYMENTS AS
SELECT c.payment_id, c.order_id, c.payment_status,
       TO_NUMBER(c.payment_amount) AS payment_amount,
       TO_TIMESTAMP_NTZ(c.paid_at, 'YYYY-MM-DD HH24:MI:SS') AS paid_at,
       c.order_id IN (SELECT order_id FROM CORE.ORDERS) AS has_valid_order,
       c.src_row
FROM CORE.PAYMENTS_CHK c WHERE c.dq_status = 'OK';

INSERT INTO CORE.ISSUES_WORK
SELECT 'payments.csv', src_row, payment_id, 'INVALID_REFERENCE', 'Parent order not found: loaded with customer_key = -1', order_id, 'FLAGGED'
FROM CORE.PAYMENTS WHERE NOT has_valid_order;
-- <<< LOGIC END (정제)

-- 확인 (기대값): REMOVED 473 / QUARANTINED 441 / FLAGGED 286 (STEP 5 이후 559)
SELECT action, error_code, COUNT(*) AS events FROM CORE.ISSUES_WORK GROUP BY action, error_code ORDER BY action, error_code;

## STEP 5. CORE 주문 품질 판정

In [ ]:
-- >>> LOGIC START
CREATE OR REPLACE TABLE CORE.ORDER_QUALITY AS
WITH blocked AS (
  SELECT DISTINCT order_id FROM CORE.ISSUES_WORK
  WHERE action IN ('QUARANTINED', 'FLAGGED') AND order_id IS NOT NULL),
items AS (
  SELECT order_id, COUNT(*) AS line_count, SUM(quantity * unit_price) AS line_amount_total
  FROM CORE.ORDER_ITEMS GROUP BY order_id),
pays AS (
  SELECT order_id,
         SUM(CASE WHEN payment_status = 'succeeded' THEN 1 ELSE 0 END)              AS success_payment_count,
         SUM(CASE WHEN payment_status = 'succeeded' THEN payment_amount ELSE 0 END) AS success_payment_amount,
         MIN(CASE WHEN payment_status = 'succeeded' THEN paid_at END)               AS first_success_paid_at
  FROM CORE.PAYMENTS WHERE has_valid_order GROUP BY order_id),
q AS (
  SELECT o.order_id, o.customer_id, o.order_date, o.order_status, o.order_amount,
         COALESCE(i.line_count, 0)             AS line_count,
         COALESCE(i.line_amount_total, 0)      AS line_amount_total,
         COALESCE(p.success_payment_count, 0)  AS success_payment_count,
         COALESCE(p.success_payment_amount, 0) AS success_payment_amount,
         p.first_success_paid_at,
         CASE
           WHEN b.order_id IS NOT NULL                    THEN 'DEPENDENCY_ERROR'
           WHEN o.order_status <> 'completed'             THEN 'NOT_COMPLETED'
           WHEN COALESCE(p.success_payment_count, 0) <> 1 THEN 'SUCCESS_PAYMENT_COUNT'
           WHEN COALESCE(i.line_count, 0) = 0             THEN 'MISSING_ITEMS'
           WHEN NOT (o.order_amount = i.line_amount_total AND i.line_amount_total = p.success_payment_amount) THEN 'AMOUNT_MISMATCH'
         END AS exclusion_reason
  FROM CORE.ORDERS o
  LEFT JOIN blocked b ON o.order_id = b.order_id
  LEFT JOIN items   i ON o.order_id = i.order_id
  LEFT JOIN pays    p ON o.order_id = p.order_id)
SELECT q.*, q.exclusion_reason IS NULL AS is_revenue_eligible FROM q;

INSERT INTO CORE.ISSUES_WORK
SELECT 'orders.csv', o.src_row, q.order_id, q.exclusion_reason, 'Order excluded from revenue (still loaded)', q.order_id, 'FLAGGED'
FROM CORE.ORDER_QUALITY q JOIN CORE.ORDERS o ON q.order_id = o.order_id
WHERE q.exclusion_reason IN ('SUCCESS_PAYMENT_COUNT', 'MISSING_ITEMS', 'AMOUNT_MISMATCH');

-- Fact 적재 직전 형태 (키는 아직 업무 키. 대리키는 STEP 7 에서 연결)
CREATE OR REPLACE TABLE CORE.SALES_LINE_PREP AS
SELECT i.order_item_id, i.order_id, q.customer_id, i.product_id, q.order_date, q.order_status,
       i.quantity, i.unit_price, i.quantity * i.unit_price AS line_amount,
       q.is_revenue_eligible, q.exclusion_reason
FROM CORE.ORDER_ITEMS i JOIN CORE.ORDER_QUALITY q ON i.order_id = q.order_id;

CREATE OR REPLACE TABLE CORE.PAYMENT_PREP AS
SELECT p.payment_id, p.order_id, o.customer_id, o.order_date, p.paid_at, CAST(p.paid_at AS DATE) AS paid_date,
       p.payment_status, p.payment_amount, p.has_valid_order
FROM CORE.PAYMENTS p LEFT JOIN CORE.ORDERS o ON p.order_id = o.order_id;

CREATE OR REPLACE TABLE CORE.RUN_SUMMARY AS
SELECT (SELECT COUNT(*) FROM RAW.V_CUSTOMERS_LATEST) + (SELECT COUNT(*) FROM RAW.V_PRODUCTS_LATEST)
     + (SELECT COUNT(*) FROM RAW.V_ORDERS_LATEST) + (SELECT COUNT(*) FROM RAW.V_ORDER_ITEMS_LATEST)
     + (SELECT COUNT(*) FROM RAW.V_PAYMENTS_LATEST)                                              AS input_rows,
       (SELECT COUNT(*) FROM CORE.ISSUES_WORK WHERE action = 'REMOVED')                          AS duplicate_rows,
       (SELECT COUNT(DISTINCT source_file || ':' || CAST(source_row_number AS VARCHAR))
          FROM CORE.ISSUES_WORK WHERE action = 'QUARANTINED')                                    AS quarantined_rows,
       (SELECT COUNT(*) FROM CORE.ISSUES_WORK WHERE action = 'FLAGGED')                          AS flagged_issues,
       (SELECT COUNT(*) FROM CORE.ORDER_QUALITY WHERE is_revenue_eligible)                       AS revenue_orders,
       (SELECT COALESCE(SUM(order_amount), 0) FROM CORE.ORDER_QUALITY WHERE is_revenue_eligible) AS revenue_amount;
-- <<< LOGIC END (판정)

-- 확인: 판정 결과 분포 (기대값: 매출 인식 27,055 / NOT_COMPLETED 2,190 / DEPENDENCY_ERROR 284 /
--        AMOUNT_MISMATCH 141 / SUCCESS_PAYMENT_COUNT 101 / MISSING_ITEMS 31). 제외된 주문도 모두 "적재"됩니다.
SELECT COALESCE(exclusion_reason, '(매출 인식)') AS result, COUNT(*) AS orders, SUM(order_amount) AS order_amount
FROM CORE.ORDER_QUALITY GROUP BY 1 ORDER BY orders DESC;

-- 원래 교육용 사례(O001~O010)는 그대로 들어 있습니다: O001·O002 만 매출 인식
SELECT order_id, order_status, order_amount, line_amount_total, success_payment_count, exclusion_reason
FROM CORE.ORDER_QUALITY WHERE order_id BETWEEN 'O001' AND 'O010' ORDER BY order_id;

## STEP 6. 감사 기록

In [ ]:
INSERT INTO AUDIT.DATA_ISSUES (load_id, source_file, source_row_number, record_key, error_code, error_detail, order_id, action)
SELECT $load_id, source_file, source_row_number, record_key, error_code, error_detail, order_id, action
FROM CORE.ISSUES_WORK;

SELECT * FROM CORE.RUN_SUMMARY;   -- 기대값: 129,618 / 473 / 441 / 559 / 27,055 / 2,198,198,700

## STEP 7. MART 반영

### 7-0. 실행 컨텍스트 복원

In [ ]:
-- 새 세션(노트북 재시작 등)이면 변수가 사라집니다. CORE.RUN_CTX 에서 다시 읽어 옵니다.
SET load_id = (SELECT load_id FROM CORE.RUN_CTX);
SET run_ts  = (SELECT run_ts  FROM CORE.RUN_CTX);
SELECT $load_id AS load_id, $run_ts AS run_ts;

In [ ]:
-- 이번 배치에 정상 주문이 있어야 MART 를 갱신합니다. 0 이면 아래 7-1 이후 셀을 실행하지 마세요.
SELECT $load_id AS load_id, $run_ts AS run_ts,
       (SELECT COUNT(*) FROM CORE.ORDERS) AS accepted_orders,
       IFF((SELECT COUNT(*) FROM CORE.ORDERS) = 0, '중단: 정상 주문 0건', 'OK') AS guard;

### 7-0. 가드 (매일 배치 전용) : 정상 주문이 0건이면 MART 를 건드리지 않고 실패로 끝냅니다.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
load_id = session.sql("SELECT load_id FROM AE101_SALES_ENT.CORE.RUN_CTX").collect()[0][0]
accepted = session.sql("SELECT COUNT(*) FROM AE101_SALES_ENT.CORE.ORDERS").collect()[0][0]

if accepted == 0:
    session.sql(
        """UPDATE AE101_SALES_ENT.AUDIT.PIPELINE_RUNS
              SET status = 'FAILED', finished_at = CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
                  message = 'Guard: no accepted orders. MART not updated.'
            WHERE load_id = ?""",
        params=[load_id],
    ).collect()
    raise RuntimeError(f"[{load_id}] 정상 주문 0건: MART 반영을 중단합니다.")

print(f"[{load_id}] guard OK - 정상 주문 {accepted:,}건")

### 7-1a. SCD2 : DIM_CUSTOMER

In [ ]:
-- 7-1. SCD2 : dim_customer
UPDATE MART.DIM_CUSTOMER
   SET valid_to = $run_ts, is_current = FALSE
  FROM CORE.CUSTOMERS s
 WHERE MART.DIM_CUSTOMER.customer_id = s.customer_id
   AND MART.DIM_CUSTOMER.is_current
   AND MART.DIM_CUSTOMER.customer_key <> -1
   AND MART.DIM_CUSTOMER.hash_diff <> s.hash_diff;

INSERT INTO MART.DIM_CUSTOMER (customer_id, customer_name, email, region, valid_from, valid_to, is_current, hash_diff, load_id)
SELECT s.customer_id, s.customer_name, s.email, s.region,
       CASE WHEN h.customer_id IS NULL THEN '1900-01-01'::TIMESTAMP_NTZ ELSE $run_ts END,  -- 처음 보는 고객은 과거 전체에 유효
       NULL, TRUE, s.hash_diff, $load_id
FROM CORE.CUSTOMERS s
LEFT JOIN (SELECT DISTINCT customer_id FROM MART.DIM_CUSTOMER) h ON h.customer_id = s.customer_id
WHERE NOT EXISTS (SELECT 1 FROM MART.DIM_CUSTOMER d WHERE d.customer_id = s.customer_id AND d.is_current);

### 7-1b. SCD2 : DIM_PRODUCT

In [ ]:
-- 7-1. SCD2 : dim_product
UPDATE MART.DIM_PRODUCT
   SET valid_to = $run_ts, is_current = FALSE
  FROM CORE.PRODUCTS s
 WHERE MART.DIM_PRODUCT.product_id = s.product_id
   AND MART.DIM_PRODUCT.is_current
   AND MART.DIM_PRODUCT.product_key <> -1
   AND MART.DIM_PRODUCT.hash_diff <> s.hash_diff;

INSERT INTO MART.DIM_PRODUCT (product_id, product_name, category, valid_from, valid_to, is_current, hash_diff, load_id)
SELECT s.product_id, s.product_name, s.category,
       CASE WHEN h.product_id IS NULL THEN '1900-01-01'::TIMESTAMP_NTZ ELSE $run_ts END,
       NULL, TRUE, s.hash_diff, $load_id
FROM CORE.PRODUCTS s
LEFT JOIN (SELECT DISTINCT product_id FROM MART.DIM_PRODUCT) h ON h.product_id = s.product_id
WHERE NOT EXISTS (SELECT 1 FROM MART.DIM_PRODUCT d WHERE d.product_id = s.product_id AND d.is_current);

### 7-2a. FACT_SALES_LINE (MERGE)

In [ ]:
-- 7-2. fact_sales_line
MERGE INTO MART.FACT_SALES_LINE AS t
USING (
  SELECT p.order_item_id, p.order_id,
         COALESCE(c.customer_key, -1)      AS customer_key,
         COALESCE(pr.product_key, -1)      AS product_key,
         COALESCE(d.date_key, -1)          AS order_date_key,
         COALESCE(s.order_status_key, -1)  AS order_status_key,
         p.order_date, p.quantity, p.unit_price, p.line_amount, p.is_revenue_eligible, p.exclusion_reason
  FROM CORE.SALES_LINE_PREP p
  LEFT JOIN MART.DIM_CUSTOMER c
         ON c.customer_id = p.customer_id AND c.customer_key <> -1
        AND CAST(p.order_date AS TIMESTAMP_NTZ) >= c.valid_from
        AND CAST(p.order_date AS TIMESTAMP_NTZ) <  COALESCE(c.valid_to, '9999-12-31'::TIMESTAMP_NTZ)
  LEFT JOIN MART.DIM_PRODUCT pr
         ON pr.product_id = p.product_id AND pr.product_key <> -1
        AND CAST(p.order_date AS TIMESTAMP_NTZ) >= pr.valid_from
        AND CAST(p.order_date AS TIMESTAMP_NTZ) <  COALESCE(pr.valid_to, '9999-12-31'::TIMESTAMP_NTZ)
  LEFT JOIN MART.DIM_DATE d          ON d.calendar_date = p.order_date
  LEFT JOIN MART.DIM_ORDER_STATUS s  ON s.status_code = p.order_status
) AS src
ON t.order_item_id = src.order_item_id
WHEN MATCHED THEN UPDATE SET
  order_id = src.order_id, customer_key = src.customer_key, product_key = src.product_key,
  order_date_key = src.order_date_key, order_status_key = src.order_status_key, order_date = src.order_date,
  quantity = src.quantity, unit_price = src.unit_price, line_amount = src.line_amount,
  is_revenue_eligible = src.is_revenue_eligible, exclusion_reason = src.exclusion_reason,
  is_deleted = FALSE, load_id = $load_id, updated_at = $run_ts
WHEN NOT MATCHED THEN INSERT
  (order_item_id, order_id, customer_key, product_key, order_date_key, order_status_key, order_date,
   quantity, unit_price, line_amount, is_revenue_eligible, exclusion_reason, is_deleted, load_id, updated_at)
VALUES
  (src.order_item_id, src.order_id, src.customer_key, src.product_key, src.order_date_key, src.order_status_key, src.order_date,
   src.quantity, src.unit_price, src.line_amount, src.is_revenue_eligible, src.exclusion_reason, FALSE, $load_id, $run_ts);

UPDATE MART.FACT_SALES_LINE SET is_deleted = TRUE, load_id = $load_id, updated_at = $run_ts
WHERE NOT is_deleted AND order_item_id NOT IN (SELECT order_item_id FROM CORE.SALES_LINE_PREP);

### 7-2b. FACT_PAYMENT (MERGE)

In [ ]:
-- 7-2. fact_payment
MERGE INTO MART.FACT_PAYMENT AS t
USING (
  SELECT p.payment_id, p.order_id,
         COALESCE(c.customer_key, -1)       AS customer_key,
         COALESCE(d.date_key, -1)           AS paid_date_key,
         COALESCE(s.payment_status_key, -1) AS payment_status_key,
         p.paid_at, p.payment_amount, p.has_valid_order
  FROM CORE.PAYMENT_PREP p
  LEFT JOIN MART.DIM_CUSTOMER c
         ON c.customer_id = p.customer_id AND c.customer_key <> -1
        AND CAST(p.order_date AS TIMESTAMP_NTZ) >= c.valid_from
        AND CAST(p.order_date AS TIMESTAMP_NTZ) <  COALESCE(c.valid_to, '9999-12-31'::TIMESTAMP_NTZ)
  LEFT JOIN MART.DIM_DATE d            ON d.calendar_date = p.paid_date
  LEFT JOIN MART.DIM_PAYMENT_STATUS s  ON s.status_code = p.payment_status
) AS src
ON t.payment_id = src.payment_id
WHEN MATCHED THEN UPDATE SET
  order_id = src.order_id, customer_key = src.customer_key, paid_date_key = src.paid_date_key,
  payment_status_key = src.payment_status_key, paid_at = src.paid_at, payment_amount = src.payment_amount,
  has_valid_order = src.has_valid_order, is_deleted = FALSE, load_id = $load_id, updated_at = $run_ts
WHEN NOT MATCHED THEN INSERT
  (payment_id, order_id, customer_key, paid_date_key, payment_status_key, paid_at, payment_amount, has_valid_order, is_deleted, load_id, updated_at)
VALUES
  (src.payment_id, src.order_id, src.customer_key, src.paid_date_key, src.payment_status_key, src.paid_at, src.payment_amount,
   src.has_valid_order, FALSE, $load_id, $run_ts);

UPDATE MART.FACT_PAYMENT SET is_deleted = TRUE, load_id = $load_id, updated_at = $run_ts
WHERE NOT is_deleted AND payment_id NOT IN (SELECT payment_id FROM CORE.PAYMENT_PREP);

### 7-2c. FACT_ORDER_LIFECYCLE (MERGE)

In [ ]:
-- 7-2. fact_order_lifecycle
MERGE INTO MART.FACT_ORDER_LIFECYCLE AS t
USING (
  SELECT q.order_id,
         COALESCE(c.customer_key, -1)      AS customer_key,
         COALESCE(d.date_key, -1)          AS order_date_key,
         COALESCE(pd.date_key, -1)         AS first_success_paid_date_key,
         COALESCE(s.order_status_key, -1)  AS order_status_key,
         q.order_date, q.first_success_paid_at, q.order_amount, q.line_count, q.line_amount_total,
         q.success_payment_count, q.success_payment_amount,
         DATEDIFF('day', q.order_date, CAST(q.first_success_paid_at AS DATE)) AS days_to_first_success_payment,
         q.is_revenue_eligible, q.exclusion_reason
  FROM CORE.ORDER_QUALITY q
  LEFT JOIN MART.DIM_CUSTOMER c
         ON c.customer_id = q.customer_id AND c.customer_key <> -1
        AND CAST(q.order_date AS TIMESTAMP_NTZ) >= c.valid_from
        AND CAST(q.order_date AS TIMESTAMP_NTZ) <  COALESCE(c.valid_to, '9999-12-31'::TIMESTAMP_NTZ)
  LEFT JOIN MART.DIM_DATE d          ON d.calendar_date  = q.order_date
  LEFT JOIN MART.DIM_DATE pd         ON pd.calendar_date = CAST(q.first_success_paid_at AS DATE)
  LEFT JOIN MART.DIM_ORDER_STATUS s  ON s.status_code    = q.order_status
) AS src
ON t.order_id = src.order_id
WHEN MATCHED THEN UPDATE SET
  customer_key = src.customer_key, order_date_key = src.order_date_key,
  first_success_paid_date_key = src.first_success_paid_date_key, order_status_key = src.order_status_key,
  order_date = src.order_date, first_success_paid_at = src.first_success_paid_at, order_amount = src.order_amount,
  line_count = src.line_count, line_amount_total = src.line_amount_total,
  success_payment_count = src.success_payment_count, success_payment_amount = src.success_payment_amount,
  days_to_first_success_payment = src.days_to_first_success_payment,
  is_revenue_eligible = src.is_revenue_eligible, exclusion_reason = src.exclusion_reason,
  is_deleted = FALSE, load_id = $load_id, updated_at = $run_ts
WHEN NOT MATCHED THEN INSERT
  (order_id, customer_key, order_date_key, first_success_paid_date_key, order_status_key, order_date, first_success_paid_at,
   order_amount, line_count, line_amount_total, success_payment_count, success_payment_amount, days_to_first_success_payment,
   is_revenue_eligible, exclusion_reason, is_deleted, load_id, updated_at)
VALUES
  (src.order_id, src.customer_key, src.order_date_key, src.first_success_paid_date_key, src.order_status_key, src.order_date,
   src.first_success_paid_at, src.order_amount, src.line_count, src.line_amount_total, src.success_payment_count,
   src.success_payment_amount, src.days_to_first_success_payment, src.is_revenue_eligible, src.exclusion_reason,
   FALSE, $load_id, $run_ts);

UPDATE MART.FACT_ORDER_LIFECYCLE SET is_deleted = TRUE, load_id = $load_id, updated_at = $run_ts
WHERE NOT is_deleted AND order_id NOT IN (SELECT order_id FROM CORE.ORDER_QUALITY);

### 7-3. 품질 검사 (17개)

In [ ]:
-- 7-3. 품질 검사 (failures 가 모두 0 이어야 COMMIT)
DELETE FROM AUDIT.DQ_RESULTS WHERE load_id = $load_id;   -- 같은 배치를 다시 실행해도 결과가 중복되지 않게
INSERT INTO AUDIT.DQ_RESULTS (load_id, check_name, failures)
SELECT $load_id, check_name, failures FROM (
  SELECT 'dim_customer_one_current_version' AS check_name, COUNT(*) AS failures
    FROM (SELECT customer_id FROM MART.DIM_CUSTOMER GROUP BY customer_id HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) <> 1) x
  UNION ALL SELECT 'dim_customer_key_unique', COUNT(*)
    FROM (SELECT customer_key FROM MART.DIM_CUSTOMER GROUP BY customer_key HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'dim_customer_valid_range', COUNT(*)
    FROM MART.DIM_CUSTOMER WHERE valid_to IS NOT NULL AND valid_to <= valid_from
  UNION ALL SELECT 'dim_product_one_current_version', COUNT(*)
    FROM (SELECT product_id FROM MART.DIM_PRODUCT GROUP BY product_id HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) <> 1) x
  UNION ALL SELECT 'dim_product_key_unique', COUNT(*)
    FROM (SELECT product_key FROM MART.DIM_PRODUCT GROUP BY product_key HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'fact_sales_line_key_unique', COUNT(*)
    FROM (SELECT order_item_id FROM MART.FACT_SALES_LINE GROUP BY order_item_id HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'fact_sales_line_customer_fk', COUNT(*)
    FROM MART.FACT_SALES_LINE f LEFT JOIN MART.DIM_CUSTOMER c ON f.customer_key = c.customer_key WHERE c.customer_key IS NULL
  UNION ALL SELECT 'fact_sales_line_product_fk', COUNT(*)
    FROM MART.FACT_SALES_LINE f LEFT JOIN MART.DIM_PRODUCT p ON f.product_key = p.product_key WHERE p.product_key IS NULL
  UNION ALL SELECT 'fact_sales_line_date_fk', COUNT(*)
    FROM MART.FACT_SALES_LINE f LEFT JOIN MART.DIM_DATE d ON f.order_date_key = d.date_key WHERE d.date_key IS NULL
  UNION ALL SELECT 'fact_sales_line_status_fk', COUNT(*)
    FROM MART.FACT_SALES_LINE f LEFT JOIN MART.DIM_ORDER_STATUS s ON f.order_status_key = s.order_status_key WHERE s.order_status_key IS NULL
  UNION ALL SELECT 'fact_sales_line_amount_calc', COUNT(*)
    FROM MART.FACT_SALES_LINE WHERE quantity <= 0 OR unit_price < 0 OR line_amount <> quantity * unit_price
  UNION ALL SELECT 'fact_payment_key_unique', COUNT(*)
    FROM (SELECT payment_id FROM MART.FACT_PAYMENT GROUP BY payment_id HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'fact_payment_status_fk', COUNT(*)
    FROM MART.FACT_PAYMENT f LEFT JOIN MART.DIM_PAYMENT_STATUS s ON f.payment_status_key = s.payment_status_key WHERE s.payment_status_key IS NULL
  UNION ALL SELECT 'fact_order_lifecycle_key_unique', COUNT(*)
    FROM (SELECT order_id FROM MART.FACT_ORDER_LIFECYCLE GROUP BY order_id HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'revenue_order_reconciliation', COUNT(*)
    FROM MART.FACT_ORDER_LIFECYCLE
   WHERE is_revenue_eligible AND NOT is_deleted
     AND (order_amount <> line_amount_total OR order_amount <> success_payment_amount OR success_payment_count <> 1)
  UNION ALL SELECT 'revenue_line_vs_order', COUNT(*)
    FROM MART.FACT_ORDER_LIFECYCLE o
    LEFT JOIN (SELECT order_id, SUM(line_amount) AS amt FROM MART.FACT_SALES_LINE WHERE NOT is_deleted GROUP BY order_id) l
           ON o.order_id = l.order_id
   WHERE o.is_revenue_eligible AND NOT o.is_deleted AND (l.amt IS NULL OR l.amt <> o.order_amount)
  UNION ALL SELECT 'revenue_requires_known_keys', COUNT(*)
    FROM MART.FACT_SALES_LINE
   WHERE is_revenue_eligible AND NOT is_deleted AND (customer_key = -1 OR product_key = -1 OR order_date_key = -1)
) checks;

In [ ]:
SELECT check_name, failures FROM AUDIT.DQ_RESULTS WHERE load_id = $load_id ORDER BY failures DESC, check_name;   -- 전부 0

### 7-4. 상태 확정 (SUCCESS / FAILED)

In [ ]:
UPDATE AUDIT.PIPELINE_RUNS p
   SET status = IFF(q.fail_cnt = 0, 'SUCCESS', 'FAILED'),
       finished_at = CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
       message = IFF(q.fail_cnt = 0, NULL, 'DQ failed: ' || q.fail_names),
       input_rows = s.input_rows, duplicate_rows = s.duplicate_rows, quarantined_rows = s.quarantined_rows,
       flagged_issues = s.flagged_issues, revenue_orders = s.revenue_orders, revenue_amount = s.revenue_amount
  FROM CORE.RUN_SUMMARY s,
       (SELECT COALESCE(SUM(failures), 0) AS fail_cnt,
               COALESCE(LISTAGG(CASE WHEN failures > 0 THEN check_name END, ', '), '') AS fail_names
          FROM AUDIT.DQ_RESULTS WHERE load_id = $load_id) q
 WHERE p.load_id = $load_id;

SELECT load_id, status, message, input_rows, revenue_orders, revenue_amount
FROM AUDIT.PIPELINE_RUNS WHERE load_id = $load_id;   -- 기대값: SUCCESS / 27,055 / 2,198,198,700

### 7-5. 게이트 (매일 배치 전용) : 상태가 SUCCESS 가 아니면 오류를 내 태스크를 실패로 표시합니다.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
load_id = session.sql("SELECT load_id FROM AE101_SALES_ENT.CORE.RUN_CTX").collect()[0][0]
row = session.sql(
    "SELECT status, message, revenue_orders, revenue_amount FROM AE101_SALES_ENT.AUDIT.PIPELINE_RUNS WHERE load_id = ?",
    params=[load_id],
).collect()[0]

if row["STATUS"] != "SUCCESS":
    raise RuntimeError(f"[{load_id}] 배치 상태 {row['STATUS']}: {row['MESSAGE']}")

print(f"[{load_id}] SUCCESS - 매출 인식 주문 {row['REVENUE_ORDERS']:,}건, 매출 {row['REVENUE_AMOUNT']:,}원")